# Full-Pol Filter + Decompose Pipeline Tutorial

This notebook demonstrates the end-to-end polarimetric SAR processing pipeline:

```
NISAR RSLC → [T3] (single-look) → RefinedLeeFilter → FullPolHAalpha / FreemanDurden3C
```

and compares the decomposition results between two preprocessing strategies:

| Strategy | Description |
|----------|-------------|
| **Boxcar** | Uniform $N\times N$ spatial averaging → decompose |
| **Refined Lee** | Single-look T3 → RefinedLeeFilter → decompose |

The Refined Lee filter preserves edges in the span image while filtering speckle within
homogeneous regions, leading to sharper boundaries in the decomposition outputs.

**Algorithms used:**
- `CoherencyMatrix` — builds boxcar-averaged $[T_3]$ from Pauli basis SLC
- `RefinedLeeFilter` — edge-preserving polarimetric speckle filter
- `FullPolHAalpha` — Cloude-Pottier H/A/α eigendecomposition
- `FreemanDurden3C` — 3-component power decomposition

**Data:** NISAR L1 RSLC quad-pol chip or synthetic fallback.

**References:**
- Lee, J.-S., Grunes, M.R., and de Grandi, G. (1999). *Polarimetric SAR speckle filtering.*
  IEEE Trans. Geoscience and Remote Sensing, 37(5), 2363–2373.
- Cloude, S.R. and Pottier, E. (1997). *A review of target decomposition theorems in radar
  polarimetry.* IEEE Trans. Geoscience and Remote Sensing, 34(2), 498–518.
- Freeman, A. and Durden, S.L. (1998). *A three-component scattering model.* IEEE Trans.
  Geoscience and Remote Sensing, 36(3), 963–973.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.filters import RefinedLeeFilter
from grdl.image_processing.decomposition import (
    CoherencyMatrix, CovarianceMatrix,
    FullPolHAalpha, FreemanDurden3C,
)
from grdk.viewers.dual_viewer import DualGeoViewer

%gui qt6
print('Imports OK')


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency     = 'A'
polarizations = 'all'
chip_size     = 1024

# Preprocessing parameters
boxcar_win    = 7    # window for boxcar strategy
rl_kernel     = 7    # Refined Lee kernel size
rl_nlook      = 1.0  # ENL for Refined Lee

In [ ]:
# =============================================================================
# Load NISAR quad-pol chip or generate synthetic fallback
# =============================================================================
def _synthetic_quad_pol(rows=512, cols=512, seed=7):
    rng = np.random.default_rng(seed)
    shh = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    shv = (0.3 * (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols)))).astype(np.complex64)
    svh = shv.copy()
    svv = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    # Bright urban strip (double-bounce)
    c0 = cols // 4
    shh[:, c0:c0+cols//8] *= 3.0
    svv[:, c0:c0+cols//8] *= -3.0
    shv[:, c0:c0+cols//8] *= 0.1
    return shh, shv, svh, svv

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations=polarizations)
    meta = reader.metadata
    N = chip_size
    r0 = max(0, (meta.rows - N) // 2)
    c0 = max(0, (meta.cols - N) // 2)
    cube = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    pol_index = {cm.polarization: i for i, cm in enumerate(meta.channel_metadata)}
    shh = cube[pol_index['HH']]
    shv = cube[pol_index['HV']]
    svh = cube[pol_index.get('VH', pol_index['HV'])]
    svv = cube[pol_index['VV']]
    print(f'NISAR quad-pol chip: {shh.shape}')
    del cube
    gc.collect()

else:
    N = chip_size
    shh, shv, svh, svv = _synthetic_quad_pol(rows=N, cols=N)
    print(f'Synthetic quad-pol: {shh.shape}')

---
## Step 1 — Build the Coherency Matrix [T3]

Two paths are computed in parallel for the comparison:

- **Boxcar path**: `CoherencyMatrix(window_size=boxcar_win)` produces a spatially-averaged
  $[T_3]$ with uniform box filtering.
- **Refined Lee path**: single-look outer product → `RefinedLeeFilter.filter_matrix()`.

In [ ]:
# =============================================================================
# Path A — Boxcar-averaged T3
# =============================================================================
channels = np.stack([shh, shv, svh, svv], axis=0)

t0 = time.perf_counter()
coh = CoherencyMatrix(window_size=boxcar_win)
t3_boxcar = coh.compute(channels)    # (3, 3, rows, cols)
print(f'Boxcar T3 ({boxcar_win}x{boxcar_win}): {time.perf_counter()-t0:.1f}s')

span_boxcar = np.real(t3_boxcar[0,0] + t3_boxcar[1,1] + t3_boxcar[2,2])
print(f'Span range: [{span_boxcar.min():.4f}, {span_boxcar.max():.4f}]')

In [ ]:
# =============================================================================
# Path B — Refined Lee on single-look T3
# =============================================================================
rlf = RefinedLeeFilter(kernel_size=rl_kernel, nlook=rl_nlook)

t0 = time.perf_counter()
t3_refined = rlf.filter_channels(shh, shv, svh, svv, matrix_type='T3')
elapsed = time.perf_counter() - t0
print(f'RefinedLeeFilter (k={rl_kernel}): {elapsed:.1f}s')

span_refined = np.real(t3_refined[0,0] + t3_refined[1,1] + t3_refined[2,2])
print(f'Span range: [{span_refined.min():.4f}, {span_refined.max():.4f}]')

In [ ]:
# =============================================================================
# GRDK viewer — Span: boxcar vs Refined Lee
# =============================================================================
def span_to_rgb(t3, pct=(2, 98)):
    span = np.real(t3[0,0] + t3[1,1] + t3[2,2])
    db = 10.0 * np.log10(np.maximum(span, 1e-10))
    lo, hi = np.percentile(db, pct)
    g = np.clip((db - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)
    return np.stack([g, g, g], axis=0)

viewer_span = DualGeoViewer()
viewer_span.set_mode('dual')
viewer_span.setWindowTitle(
    f'Span — Boxcar {boxcar_win}x{boxcar_win} (L) vs RefinedLee k={rl_kernel} (R)'
)
viewer_span.set_array(span_to_rgb(t3_boxcar), pane=0)
viewer_span.set_array(span_to_rgb(t3_refined), pane=1)
viewer_span.show()
print('Viewer: span comparison')

---
## Step 2 — FullPolHAalpha: Boxcar vs Refined Lee

The `decompose_from_t3(t3)` interface bypasses the internal matrix build step,
allowing the same decomposer to accept either the boxcar or Refined Lee $[T_3]$.

In [ ]:
# =============================================================================
# FullPolHAalpha — boxcar path
# =============================================================================
ha = FullPolHAalpha(window_size=1)   # window=1: matrix already averaged
ha_boxcar  = ha.decompose_from_t3(t3_boxcar)
ha_refined = ha.decompose_from_t3(t3_refined)

print('Boxcar H/A/α statistics:')
print(f'  H mean={ha_boxcar["entropy"].mean():.3f}, α mean={ha_boxcar["alpha"].mean():.1f}°')
print('Refined Lee H/A/α statistics:')
print(f'  H mean={ha_refined["entropy"].mean():.3f}, α mean={ha_refined["alpha"].mean():.1f}°')

rgb_ha_box, _ = ha.to_rgb(ha_boxcar)
rgb_ha_rl,  _ = ha.to_rgb(ha_refined)

In [ ]:
# =============================================================================
# GRDK viewer — H/A/α: boxcar vs Refined Lee
# =============================================================================
viewer_ha = DualGeoViewer()
viewer_ha.set_mode('dual')
viewer_ha.setWindowTitle(
    f'FullPolHAalpha — Boxcar {boxcar_win}x{boxcar_win} (L) vs RefinedLee (R)'
)
viewer_ha.set_array(rgb_ha_box, pane=0)
viewer_ha.set_array(rgb_ha_rl, pane=1)
viewer_ha.show()
print('Viewer: FullPolHAalpha comparison')

---
## Step 3 — FreemanDurden3C: Boxcar vs Refined Lee

`FreemanDurden3C.decompose_from_c3(c3)` accepts a pre-computed $[C_3]$ matrix.  Since
`RefinedLeeFilter` can also produce $[C_3]$ directly (`matrix_type='C3'`), both paths
stay within the pre-computed matrix interface.

In [ ]:
# =============================================================================
# FreemanDurden3C — build C3 paths and decompose
# =============================================================================
# Boxcar C3
cov = CovarianceMatrix(window_size=boxcar_win)
c3_boxcar = cov.compute(channels)

# Refined Lee C3
c3_refined = rlf.filter_channels(shh, shv, svh, svv, matrix_type='C3')

fd = FreemanDurden3C(window_size=1)
fd_boxcar  = fd.decompose_from_c3(c3_boxcar)
fd_refined = fd.decompose_from_c3(c3_refined)

print('FreemanDurden3C power budgets:')
for name, comp in [('Boxcar', fd_boxcar), ('Refined Lee', fd_refined)]:
    total = (np.nanmean(comp['surface']) + np.nanmean(comp['double_bounce']) +
             np.nanmean(comp['volume']))
    print(f'  {name}: Ps={np.nanmean(comp["surface"]):.4f}, '
          f'Pd={np.nanmean(comp["double_bounce"]):.4f}, '
          f'Pv={np.nanmean(comp["volume"]):.4f}  total={total:.4f}')

rgb_fd_box, _ = fd.to_rgb(fd_boxcar)
rgb_fd_rl,  _ = fd.to_rgb(fd_refined)

In [ ]:
# =============================================================================
# GRDK viewer — FreemanDurden: boxcar vs Refined Lee
# =============================================================================
viewer_fd = DualGeoViewer()
viewer_fd.set_mode('dual')
viewer_fd.setWindowTitle(
    f'FreemanDurden3C — Boxcar {boxcar_win}x{boxcar_win} (L) vs RefinedLee (R)'
)
viewer_fd.set_array(rgb_fd_box, pane=0)
viewer_fd.set_array(rgb_fd_rl, pane=1)
viewer_fd.show()
print('Viewer: FreemanDurden3C comparison')

---
## Summary

The cell below prints a statistical summary comparing the two preprocessing paths across
entropy, alpha, and the three Freeman-Durden power components.

In [ ]:
# =============================================================================
# Summary statistics
# =============================================================================
rows_out = [
    ('Entropy H',    ha_boxcar['entropy'],          ha_refined['entropy']),
    ('Alpha α (°)',  ha_boxcar['alpha'],             ha_refined['alpha']),
    ('Anisotropy A', ha_boxcar['anisotropy'],        ha_refined['anisotropy']),
    ('FD Ps',        fd_boxcar['surface'],           fd_refined['surface']),
    ('FD Pd',        fd_boxcar['double_bounce'],     fd_refined['double_bounce']),
    ('FD Pv',        fd_boxcar['volume'],            fd_refined['volume']),
]

hdr = f'{"Metric":<18} {"Boxcar mean":>14} {"RefinedLee mean":>16} {"Δ mean":>10}'
print(hdr)
print('-' * len(hdr))
for label, box_arr, rl_arr in rows_out:
    bm = float(np.nanmean(box_arr))
    rm = float(np.nanmean(rl_arr))
    print(f'{label:<18} {bm:>14.4f} {rm:>16.4f} {rm-bm:>10.4f}')